In [ ]:
# ==============================================================================
# CELL 0: SETUP & DATASET GENERATION (retail_sales.csv)
# Run this cell first to generate the dataset directly inside Google Colab!
# ==============================================================================
import numpy as np
import pandas as pd

# Create a realistic retail sales dataset
np.random.seed(42)
n_rows = 150

categories = ['Electronics', 'Clothing', 'Home & Kitchen', 'Beauty', 'Books']
cities = ['New York', 'Los Angeles', 'Chicago', 'Houston', 'Miami']
payment_methods = ['Credit Card', 'Debit Card', 'Cash', 'UPI', 'PayPal']
genders = ['Male', 'Female', 'Other']

dates = pd.date_range(start='2024-01-01', end='2024-06-30', periods=n_rows).strftime('%Y-%m-%d')
order_ids = [f"ORD{1000 + i}" for i in range(n_rows)]
customer_ids = [f"CUST{np.random.randint(100, 160)}" for _ in range(n_rows)]
ages = np.random.randint(18, 65, size=n_rows)
gender_col = np.random.choice(genders, size=n_rows, p=[0.48, 0.48, 0.04])
cat_col = np.random.choice(categories, size=n_rows)
qty_col = np.random.randint(1, 6, size=n_rows)
unit_prices = np.random.choice([15.0, 25.5, 49.99, 89.0, 120.0, 299.99, 450.0, 899.0], size=n_rows)
total_amounts = np.round(qty_col * unit_prices, 2)
payments = np.random.choice(payment_methods, size=n_rows)
city_col = np.random.choice(cities, size=n_rows)

raw_df = pd.DataFrame({
    'Order_ID': order_ids,
    'Date': dates,
    'Customer_ID': customer_ids,
    'Gender': gender_col,
    'Age': ages,
    'Category': cat_col,
    'Quantity': qty_col,
    'Unit_Price': unit_prices,
    'Total_Amount': total_amounts,
    'Payment_Method': payments,
    'City': city_col
})

# Inject a few missing values & sentinel strings to demonstrate cleaning
raw_df.loc[10, 'Age'] = np.nan
raw_df.loc[25, 'Payment_Method'] = "MISSING"

raw_df.to_csv("retail_sales.csv", index=False)
print("`retail_sales.csv` successfully created on disk!")

`retail_sales.csv` successfully created on disk!


In [ ]:
# ==============================================================================
# SECTION 1: SERIES VS DATAFRAME & INDEX AS A FIRST-CLASS CITIZEN
# ==============================================================================
# 1. Series: 1D labeled homogeneous array
s = pd.Series([100, 250, 375], index=['store_A', 'store_B', 'store_C'], name="Revenue")
print("--- Pandas Series ---")
print(s)
print(f"Index: {s.index}, Values: {s.values}\n")

# 2. DataFrame: 2D labeled tabular data structure (Collection of aligned Series sharing an Index)
df_toy = pd.DataFrame({
    'Revenue': [100, 250, 375],
    'Expenses': [60, 140, 200]
}, index=['store_A', 'store_B', 'store_C'])

print("--- Pandas DataFrame ---")
print(df_toy)
print(f"Row Index: {df_toy.index.tolist()} | Columns: {df_toy.columns.tolist()}")

# Index Alignment Concept:
# When performing operations between Series, Pandas matches by INDEX LABEL, not position.
bonus = pd.Series([10, 20], index=['store_B', 'store_A'])
print("\nAutomatic Label Alignment (Revenue + Bonus):")
print(df_toy['Revenue'] + bonus) # Notice store_A got 10 and store_B got 20!

--- Pandas Series ---
store_A    100
store_B    250
store_C    375
Name: Revenue, dtype: int64
Index: Index(['store_A', 'store_B', 'store_C'], dtype='object'), Values: [100 250 375]

--- Pandas DataFrame ---
         Revenue  Expenses
store_A      100        60
store_B      250       140
store_C      375       200
Row Index: ['store_A', 'store_B', 'store_C'] | Columns: ['Revenue', 'Expenses']

Automatic Label Alignment (Revenue + Bonus):
store_A    120.0
store_B    260.0
store_C      NaN
dtype: float64


In [ ]:
# ==============================================================================
# SECTION 2: CREATING DATAFRAMES FROM PYTHON & NUMPY STRUCTURES
# ==============================================================================
# 1. From Dictionary of Lists/Arrays
df_from_dict = pd.DataFrame({
    'Product': ['Laptop', 'Mouse', 'Keyboard'],
    'Stock': [15, 120, 45]
})

# 2. From List of Dictionaries (Row-oriented)
df_from_records = pd.DataFrame([
    {'Product': 'Monitor', 'Stock': 28},
    {'Product': 'Desk', 'Stock': 10}
])

# 3. From a 2D NumPy Array
np_matrix = np.random.randint(10, 99, size=(3, 3))
df_from_numpy = pd.DataFrame(
    data=np_matrix,
    index=['Row_1', 'Row_2', 'Row_3'],
    columns=['Col_A', 'Col_B', 'Col_C']
)
print("DataFrame from NumPy:\n", df_from_numpy)

DataFrame from NumPy:
        Col_A  Col_B  Col_C
Row_1     81     63     76
Row_2     60     17     43
Row_3     44     97     87


In [ ]:
# ==============================================================================
# SECTION 3: READING DATA (read_csv, read_excel, read_json)
# ==============================================================================
# pd.read_csv key parameters:
# - sep: delimiter (default ',')
# - header: row number to use as column names (default 0)
# - names: custom list of column names to override
# - usecols: list of column names or indices to load (saves memory)
# - nrows: number of rows to read from top
# - na_values: additional strings to recognize as NaN/null
# - parse_dates: list of columns to convert to datetime

retail_df = pd.read_csv(
    'retail_sales.csv',
    sep=',',
    usecols=['Order_ID', 'Date', 'Customer_ID', 'Gender', 'Age', 'Category',
             'Quantity', 'Unit_Price', 'Total_Amount', 'Payment_Method', 'City'],
    na_values=['MISSING', 'N/A', 'null'],
    parse_dates=['Date']
)

print(f"Loaded {retail_df.shape[0]} rows and {retail_df.shape[1]} columns.\n")

# Syntax references for other formats:
# df_excel = pd.read_excel('file.xlsx', sheet_name='Sheet1')
# df_json  = pd.read_json('file.json')

Loaded 150 rows and 11 columns.



In [ ]:
# ==============================================================================
# SECTION 4: FIRST-LOOK PROFILING METHODS (Updated for Pandas 2.0+)
# ==============================================================================
print("1. First 3 rows (.head):")
display(retail_df.head(3))

print("\n2. Dimensions (.shape):", retail_df.shape)
print("3. Data types (.dtypes):\n", retail_df.dtypes)

print("\n4. Structural summary (.info):")
retail_df.info()

print("\n5. Statistical distribution (.describe):")
display(retail_df.describe(include='all'))

print("\n6. Unique counts per column (.nunique):")
print(retail_df.nunique())

print("\n7. Frequency counts for Category (.value_counts):")
print(retail_df['Category'].value_counts(normalize=False))

1. First 3 rows (.head):


,Order_ID,Date,Customer_ID,Gender,Age,Category,Quantity,Unit_Price,Total_Amount,Payment_Method,City
0,ORD1000,2024-01-01,CUST138,Female,50.0,Electronics,4,49.99,199.96,Cash,Miami
1,ORD1001,2024-01-02,CUST151,Female,40.0,Books,4,450.00,1800.00,Cash,Miami
2,ORD1002,2024-01-03,CUST128,Male,41.0,Clothing,3,299.99,899.97,Credit Card,Chicago



2. Dimensions (.shape): (150, 11)
3. Data types (.dtypes):
 Order_ID                  object
Date              datetime64[ns]
Customer_ID               object
Gender                    object
Age                      float64
Category                  object
Quantity                   int64
Unit_Price               float64
Total_Amount             float64
Payment_Method            object
City                      object
dtype: object

4. Structural summary (.info):
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   Order_ID        150 non-null    object        
 1   Date            150 non-null    datetime64[ns]
 2   Customer_ID     150 non-null    object        
 3   Gender          150 non-null    object        
 4   Age             149 non-null    float64       
 5   Category        150 non-null    object        
 6   Quan

,Order_ID,Date,Customer_ID,Gender,Age,Category,Quantity,Unit_Price,Total_Amount,Payment_Method,City
count,150,150,150,150,149.000000,150,150.000000,150.000000,150.000000,149,150
unique,150,NaN,56,3,NaN,5,NaN,NaN,NaN,5,5
top,ORD1000,NaN,CUST159,Female,NaN,Books,NaN,NaN,NaN,Credit Card,Chicago
freq,1,NaN,8,80,NaN,35,NaN,NaN,NaN,36,32
mean,NaN,2024-03-31 00:09:36,NaN,NaN,40.610738,NaN,3.080000,242.463933,743.896133,NaN,NaN
min,NaN,2024-01-01 00:00:00,NaN,NaN,18.000000,NaN,1.000000,15.000000,15.000000,NaN,NaN
25%,NaN,2024-02-14 12:00:00,NaN,NaN,29.000000,NaN,2.000000,25.500000,91.745000,NaN,NaN
50%,NaN,2024-03-31 00:00:00,NaN,NaN,43.000000,NaN,3.000000,104.500000,299.990000,NaN,NaN
75%,NaN,2024-05-15 12:00:00,NaN,NaN,52.000000,NaN,4.750000,412.497500,899.727500,NaN,NaN
max,NaN,2024-06-30 00:00:00,NaN,NaN,64.000000,NaN,5.000000,899.000000,4495.000000,NaN,NaN



6. Unique counts per column (.nunique):
Order_ID          150
Date              150
Customer_ID        56
Gender              3
Age                44
Category            5
Quantity            5
Unit_Price          8
Total_Amount       40
Payment_Method      5
City                5
dtype: int64

7. Frequency counts for Category (.value_counts):
Category
Books             35
Electronics       31
Clothing          31
Beauty            28
Home & Kitchen    25
Name: count, dtype: int64


In [ ]:
# ==============================================================================
# SECTION 5: SELECTION & CONTRAST: .loc vs .iloc
# ==============================================================================
# Setup an explicit string index for clear illustration
sample_df = retail_df.head(5).copy()
sample_df.index = ['idx_a', 'idx_b', 'idx_c', 'idx_d', 'idx_e']
print("Sample DataFrame with Custom Index:\n", sample_df[['Customer_ID', 'Category', 'Total_Amount']])

# --- 1. Column Selection ---
s_col = sample_df['Category']       # Returns a Series (1D)
df_cols = sample_df[['Category']]   # Returns a DataFrame (2D)

# --- 2. .loc (LABEL-BASED) ---
# Syntax: df.loc[row_label_start:row_label_stop, col_name_start:col_name_stop]
# RULE: .loc endpoint IS INCLUSIVE!
loc_result = sample_df.loc['idx_a':'idx_c', ['Category', 'Total_Amount']]
print("\n--- .loc Result ('idx_a' through 'idx_c' inclusive) ---")
print(loc_result)

# --- 3. .iloc (INTEGER POSITION-BASED, 0-indexed) ---
# Syntax: df.iloc[row_idx_start:row_idx_stop, col_idx_start:col_idx_stop]
# RULE: .iloc endpoint IS EXCLUSIVE (standard Python slicing)!
iloc_result = sample_df.iloc[0:2, [5, 8]] # Rows 0 and 1, cols 5 (Category) and 8 (Total_Amount)
print("\n--- .iloc Result (Rows 0:2 exclusive) ---")
print(iloc_result)

Sample DataFrame with Custom Index:
       Customer_ID     Category  Total_Amount
idx_a     CUST138  Electronics        199.96
idx_b     CUST151        Books       1800.00
idx_c     CUST128     Clothing        899.97
idx_d     CUST114     Clothing       4495.00
idx_e     CUST142        Books        356.00

--- .loc Result ('idx_a' through 'idx_c' inclusive) ---
          Category  Total_Amount
idx_a  Electronics        199.96
idx_b        Books       1800.00
idx_c     Clothing        899.97

--- .iloc Result (Rows 0:2 exclusive) ---
          Category  Total_Amount
idx_a  Electronics        199.96
idx_b        Books       1800.00


In [ ]:
# ==============================================================================
# SECTION 6: CONDITIONAL FILTERING & QUERYING
# ==============================================================================
# Rule: Use & (AND), | (OR), ~ (NOT). ALWAYS wrap individual conditions in parentheses!

# 1. Single condition
cheap_items = retail_df[retail_df['Unit_Price'] < 30]

# 2. Multiple conditions (& / |)
high_val_ny = retail_df[(retail_df['City'] == 'New York') & (retail_df['Total_Amount'] > 500)]
print(f"High-value transactions in New York: {len(high_val_ny)}")

# 3. .isin() - cleaner alternative to chained OR statements
selected_cities = retail_df[retail_df['City'].isin(['Chicago', 'Miami', 'Houston'])]

# 4. .between(low, high) - inclusive range check
young_adults = retail_df[retail_df['Age'].between(18, 25)]

# 5. .query() - SQL-like string syntax (can reference Python variables with @)
target_cat = 'Electronics'
query_result = retail_df.query("Category == @target_cat and Total_Amount >= 1000")
print(f"Electronics orders >= $1000: {len(query_result)}")

High-value transactions in New York: 14
Electronics orders >= $1000: 2


In [ ]:
# ==============================================================================
# SECTION 7: ADDING, RENAMING, DROPPING, SORTING, & INDEX MANIPULATION
# ==============================================================================
df_mod = retail_df.copy()

# 1. Adding a new derived column
df_mod['Discounted_Total'] = df_mod['Total_Amount'] * 0.90 # 10% discount

# 2. Renaming columns
df_mod = df_mod.rename(columns={'Total_Amount': 'Gross_Revenue', 'City': 'Location'})

# 3. Dropping columns and rows
df_mod = df_mod.drop(columns=['Customer_ID']) # Drop column
df_mod = df_mod.drop(index=[0, 1])            # Drop first two rows

# 4. Sorting
sorted_df = df_mod.sort_values(by=['Location', 'Gross_Revenue'], ascending=[True, False])

# 5. set_index / reset_index
df_indexed = sorted_df.set_index('Order_ID')
print("Index set to Order_ID:\n", df_indexed.head(2))

df_restored = df_indexed.reset_index() # Restores Order_ID back as a column

Index set to Order_ID:
                Date Gender   Age        Category  Quantity  Unit_Price  \
Order_ID                                                                 
ORD1032  2024-02-08   Male  47.0  Home & Kitchen         5       899.0   
ORD1115  2024-05-19   Male  23.0           Books         5       899.0   

          Gross_Revenue Payment_Method Location  Discounted_Total  
Order_ID                                                           
ORD1032          4495.0           Cash  Chicago            4045.5  
ORD1115          4495.0         PayPal  Chicago            4045.5  


In [ ]:
# ==============================================================================
# SECTION 8: LIVE CODE - RETAIL BUSINESS ANALYSIS
# ==============================================================================
print("=== LIVE-CODE DEMO: RETAIL BUSINESS INSIGHTS ===")

# --- Business Question 1 ---
# "Which Electronics transactions exceeded $500 and were paid with a Credit Card?"
q1 = retail_df[
    (retail_df['Category'] == 'Electronics') &
    (retail_df['Total_Amount'] > 500) &
    (retail_df['Payment_Method'] == 'Credit Card')
][['Order_ID', 'Date', 'Customer_ID', 'Total_Amount', 'Payment_Method']]
print("\nQ1: High-Value Electronics (Credit Card):")
display(q1.head())

# --- Business Question 2 ---
# "Find top 5 highest-revenue transactions made by customers aged 25-35 in 'New York' or 'Chicago' buying 'Clothing' or 'Beauty'."
q2 = retail_df[
    (retail_df['Age'].between(25, 35)) &
    (retail_df['City'].isin(['New York', 'Chicago'])) &
    (retail_df['Category'].isin(['Clothing', 'Beauty']))
].sort_values(by='Total_Amount', ascending=False).head(5)
print("\nQ2: Top 5 Targeted Demographic Transactions:")
display(q2[['Order_ID', 'Age', 'City', 'Category', 'Quantity', 'Total_Amount']])

# --- Business Question 3 ---
# "Find all transactions where the Unit_Price is higher than the overall average Unit_Price, sorted by Date descending."
avg_unit_price = retail_df['Unit_Price'].mean()
q3 = retail_df[retail_df['Unit_Price'] > avg_unit_price].sort_values(by='Date', ascending=False)
print(f"\nQ3: Above-Average Unit Price (Avg = ${avg_unit_price:.2f}) - Total records: {len(q3)}:")
display(q3[['Order_ID', 'Date', 'Category', 'Unit_Price', 'Total_Amount']].head())

=== LIVE-CODE DEMO: RETAIL BUSINESS INSIGHTS ===

Q1: High-Value Electronics (Credit Card):


,Order_ID,Date,Customer_ID,Total_Amount,Payment_Method
54,ORD1054,2024-03-06,CUST108,1800.0,Credit Card
130,ORD1130,2024-06-06,CUST123,899.0,Credit Card



Q2: Top 5 Targeted Demographic Transactions:


,Order_ID,Age,City,Category,Quantity,Total_Amount
24,ORD1024,32.0,Chicago,Beauty,4,1800.00
41,ORD1041,30.0,New York,Clothing,3,899.97
22,ORD1022,26.0,Chicago,Clothing,4,356.00
68,ORD1068,25.0,New York,Clothing,4,199.96
135,ORD1135,28.0,Chicago,Clothing,2,99.98



Q3: Above-Average Unit Price (Avg = $242.46) - Total records: 58:


,Order_ID,Date,Category,Unit_Price,Total_Amount
149,ORD1149,2024-06-30,Beauty,299.99,899.97
144,ORD1144,2024-06-23,Electronics,450.00,450.00
142,ORD1142,2024-06-21,Home & Kitchen,299.99,1499.95
141,ORD1141,2024-06-20,Books,450.00,1800.00
140,ORD1140,2024-06-19,Books,450.00,450.00


In [ ]:
# ==============================================================================
# SECTION 9: 5 STUDENT PRACTICE EXERCISES
# ==============================================================================
print("=== STUDENT INDEPENDENT EXERCISES ===")

# --- Exercise 1 ---
# Find all transactions where Quantity is at least 4 and Payment_Method is 'Cash' or 'UPI'.
# YOUR CODE HERE:
ex1 = retail_df[(retail_df['Quantity'] >= 4) & (retail_df['Payment_Method'].isin(['Cash', 'UPI']))]
print(f"Exercise 1 count: {len(ex1)}")


# --- Exercise 2 ---
# Extract only the 'Customer_ID', 'Category', and 'Total_Amount' columns for all transactions in 'Miami' with Total_Amount < 100.
# YOUR CODE HERE:
ex2 = retail_df[(retail_df['City'] == 'Miami') & (retail_df['Total_Amount'] < 100)][['Customer_ID', 'Category', 'Total_Amount']]
print(f"Exercise 2 count: {len(ex2)}")


# --- Exercise 3 ---
# Using .query(), find all female customers ('Female') who purchased items in the 'Books' or 'Home & Kitchen' category.
# YOUR CODE HERE:
ex3 = retail_df.query("Gender == 'Female' and Category in ['Books', 'Home & Kitchen']")
print(f"Exercise 3 count: {len(ex3)}")


# --- Exercise 4 ---
# Identify the top 3 highest Total_Amount orders for senior customers (Age >= 50) and return only Order_ID, Age, Category, and Total_Amount.
# YOUR CODE HERE:
ex4 = (
    retail_df[retail_df['Age'] >= 50]
    .sort_values(by='Total_Amount', ascending=False)
    [['Order_ID', 'Age', 'Category', 'Total_Amount']]
    .head(3)
)
print("\nExercise 4 top 3:")
display(ex4)


# --- Exercise 5 ---
# Filter all rows where Age is NOT missing (NaN) and Total_Amount is between $200 and $800 (inclusive).
# Sort the result first by Category (A-Z) and then by Total_Amount (descending).
# YOUR CODE HERE:
ex5 = (
    retail_df[retail_df['Age'].notna() & retail_df['Total_Amount'].between(200, 800)]
    .sort_values(by=['Category', 'Total_Amount'], ascending=[True, False])
)
print(f"\nExercise 5 count: {len(ex5)}")
display(ex5.head())

=== STUDENT INDEPENDENT EXERCISES ===
Exercise 1 count: 26
Exercise 2 count: 6
Exercise 3 count: 32

Exercise 4 top 3:


,Order_ID,Age,Category,Total_Amount
3,ORD1003,54.0,Clothing,4495.0
38,ORD1038,60.0,Books,4495.0
48,ORD1048,62.0,Books,4495.0



Exercise 5 count: 42


,Order_ID,Date,Customer_ID,Gender,Age,Category,Quantity,Unit_Price,Total_Amount,Payment_Method,City
78,ORD1078,2024-04-04,CUST135,Female,49.0,Beauty,5,120.00,600.00,Credit Card,Houston
37,ORD1037,2024-02-14,CUST158,Female,32.0,Beauty,4,89.00,356.00,Credit Card,Los Angeles
18,ORD1018,2024-01-22,CUST102,Female,43.0,Beauty,1,299.99,299.99,Credit Card,Houston
129,ORD1129,2024-06-05,CUST107,Female,50.0,Beauty,1,299.99,299.99,Credit Card,New York
77,ORD1077,2024-04-03,CUST116,Male,49.0,Books,5,120.00,600.00,Cash,Chicago
